# Retinal OCT Layer Semantic Segmentation

**Pipeline overview:**
1. Load the pretrained 8-layer segmentation model
2. Run inference on all input images → auto-generate 3-class GT masks
3. Train a new U-Net with the generated labels
4. Evaluate and visualize results

**Class mapping (8-layer → 3-class):**
- Class 0 (Background): layer 0
- Class 1 (Inner Retina – NFL, GCL+IPL, INL): layers 1, 2, 3
- Class 2 (Outer Retina – OPL, ONL, Ellipsoid zone): layers 4, 5, 6
- Class 3 (RPE + Choroid): layers 7, 8

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, sys, random
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from keras.utils import normalize, to_categorical
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

# ── paths ──────────────────────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/Retina_layer_semantic_segmentation/data'
WEIGHTS_PATH = '/content/drive/MyDrive/Retina_layer_semantic_segmentation/retina_segmentation_8_layer.hdf5'

SIZE_X = 640          # image width  (resize target)
SIZE_Y = 640          # image height (resize target)
N_CLASSES = 4         # background + 3 retinal classes
BATCH_SIZE = 4
EPOCHS = 50

# colour map for overlay visualisation
CLASS_COLORS = [
    (0,   0,   0),    # 0 – background  (black)
    (255, 0,   0),    # 1 – Inner Retina (red)
    (0,   255, 0),    # 2 – Outer Retina (green)
    (0,   0,   255),  # 3 – RPE + Choroid (blue)
]

## Step 1 – Load images

In [ ]:
image_paths = []
for group in ['group1', 'group2', 'group3']:
    image_paths.extend(glob.glob(os.path.join(DRIVE_ROOT, group, '*.jpeg')))
    image_paths.extend(glob.glob(os.path.join(DRIVE_ROOT, group, '*.jpg')))
    image_paths.extend(glob.glob(os.path.join(DRIVE_ROOT, group, '*.png')))

image_paths = sorted(image_paths)
print(f'Total images found: {len(image_paths)}')

raw_images = []
for p in image_paths:
    img = cv2.imread(p, 0)                      # grayscale
    img = cv2.resize(img, (SIZE_X, SIZE_Y))
    raw_images.append(img)

raw_images = np.array(raw_images)               # (N, H, W)  uint8
print('raw_images shape:', raw_images.shape)

In [ ]:
# Normalise for model input  →  (N, H, W, 1)  float32
images_norm = np.expand_dims(raw_images, axis=3).astype(np.float32)
images_norm = normalize(images_norm, axis=1)
print('images_norm shape:', images_norm.shape)

In [ ]:
# Figure 1 – Sample Input Images
n_show = min(6, len(raw_images))
fig, axes = plt.subplots(1, n_show, figsize=(18, 4))
fig.suptitle('Figure 1 – Sample Input Images (Retinal OCT)', fontsize=14)
for i, ax in enumerate(axes):
    ax.imshow(raw_images[i], cmap='gray')
    ax.set_title(f'Image {i+1}')
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/figure1_sample_inputs.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 2 – Load pretrained 8-layer model & generate 3-class GT masks

In [ ]:
from simple_unet import multi_unet_model

pretrained = multi_unet_model(n_classes=9, IMG_HEIGHT=SIZE_Y, IMG_WIDTH=SIZE_X, IMG_CHANNELS=1)
pretrained.load_weights(WEIGHTS_PATH)
print('Pretrained 8-layer model loaded.')

In [ ]:
def convert_8_to_3(mask: np.ndarray) -> np.ndarray:
    """Map 9-class (0–8) prediction to 4-class (0–3).

    0          → 0  Background
    1, 2, 3    → 1  Inner Retina  (NFL, GCL+IPL, INL)
    4, 5, 6    → 2  Outer Retina  (OPL, ONL, Ellipsoid zone)
    7, 8       → 3  RPE + Choroid
    """
    out = np.zeros(mask.shape, dtype=np.uint8)
    out[np.isin(mask, [1, 2, 3])] = 1
    out[np.isin(mask, [4, 5, 6])] = 2
    out[np.isin(mask, [7, 8])]    = 3
    return out

In [ ]:
generated_masks = []

for i in range(len(images_norm)):
    img_input = np.expand_dims(images_norm[i], axis=0)   # (1, H, W, 1)
    pred      = pretrained.predict(img_input, verbose=0)  # (1, H, W, 9)
    pred_mask = np.argmax(pred, axis=3)[0]                # (H, W)
    mask3     = convert_8_to_3(pred_mask)
    generated_masks.append(mask3)

generated_masks = np.array(generated_masks)   # (N, H, W)  uint8
print('generated_masks shape:', generated_masks.shape)
print('unique class values  :', np.unique(generated_masks))

In [ ]:
# Figure 2 – Auto-generated 3-class GT Masks
n_show = min(6, len(generated_masks))
fig, axes = plt.subplots(1, n_show, figsize=(18, 4))
fig.suptitle('Figure 2 – Auto-generated 3-class GT Masks', fontsize=14)
cmap_seg = plt.cm.get_cmap('tab10', N_CLASSES)
for i, ax in enumerate(axes):
    ax.imshow(generated_masks[i], cmap=cmap_seg, vmin=0, vmax=N_CLASSES-1)
    ax.set_title(f'Image {i+1}')
    ax.axis('off')
patches = [mpatches.Patch(color=cmap_seg(c), label=['Background','Inner Retina','Outer Retina','RPE+Choroid'][c]) for c in range(N_CLASSES)]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=10)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig('/content/figure2_gt_masks.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 3 – Prepare train / test split & one-hot encoding

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    images_norm, generated_masks,
    test_size=0.15, random_state=42
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Unique classes in y_train: {np.unique(y_train)}')

# One-hot encode masks   →  (N, H, W, 4)
y_train_cat = to_categorical(y_train, num_classes=N_CLASSES)
y_test_cat  = to_categorical(y_test,  num_classes=N_CLASSES)
print('y_train_cat shape:', y_train_cat.shape)

## Step 4 – Define and train the 3-class U-Net

In [ ]:
model = multi_unet_model(
    n_classes=N_CLASSES,
    IMG_HEIGHT=SIZE_Y,
    IMG_WIDTH=SIZE_X,
    IMG_CHANNELS=1
)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/Retina_layer_semantic_segmentation/retina_3class_best.hdf5',
    monitor='val_loss', save_best_only=True, verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
)

In [ ]:
history = model.fit(
    X_train, y_train_cat,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_test, y_test_cat),
    callbacks=[checkpoint, early_stop, reduce_lr],
    shuffle=True,
    verbose=1
)

_, acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'Test accuracy: {acc*100:.2f} %')

## Step 5 – Learning curves

In [ ]:
# Figure 3 – Training and Validation Loss
epochs_range = range(1, len(history.history['loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Figure 3 – Learning Curves', fontsize=14)

axes[0].plot(epochs_range, history.history['loss'],     'b-o', markersize=3, label='Train Loss')
axes[0].plot(epochs_range, history.history['val_loss'], 'r-o', markersize=3, label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Categorical Cross-Entropy Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history.history['accuracy'],     'b-o', markersize=3, label='Train Accuracy')
axes[1].plot(epochs_range, history.history['val_accuracy'], 'r-o', markersize=3, label='Val Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Pixel Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/figure3_learning_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## Step 6 – Evaluation: Mean IoU

In [ ]:
from keras.metrics import MeanIoU

y_pred        = model.predict(X_test, verbose=0)
y_pred_argmax = np.argmax(y_pred, axis=3)          # (N, H, W)

iou_metric = MeanIoU(num_classes=N_CLASSES)
iou_metric.update_state(y_test, y_pred_argmax)
print(f'Mean IoU: {iou_metric.result().numpy():.4f}')

cm = np.array(iou_metric.get_weights()).reshape(N_CLASSES, N_CLASSES)
class_names = ['Background', 'Inner Retina', 'Outer Retina', 'RPE+Choroid']
for c in range(N_CLASSES):
    tp  = cm[c, c]
    fn  = cm[c, :].sum() - tp
    fp  = cm[:, c].sum() - tp
    iou = tp / (tp + fn + fp + 1e-7)
    print(f'  IoU [{class_names[c]}]: {iou:.4f}')

## Step 7 – Qualitative results: Input / GT Mask / Prediction / Overlay

In [ ]:
def make_overlay(gray_img: np.ndarray, mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """Blend a grayscale image with a colour-coded segmentation mask."""
    rgb = cv2.cvtColor(gray_img, cv2.COLOR_GRAY2RGB).astype(np.float32)
    colour_mask = np.zeros_like(rgb)
    for cls_idx, colour in enumerate(CLASS_COLORS):
        colour_mask[mask == cls_idx] = colour
    blended = cv2.addWeighted(rgb, 1 - alpha, colour_mask, alpha, 0)
    return blended.astype(np.uint8)

In [ ]:
# Figure 4 – Segmentation Results (Input | GT Mask | Prediction | Overlay)
n_show   = min(5, len(X_test))
fig, axes = plt.subplots(n_show, 4, figsize=(20, n_show * 5))
fig.suptitle('Figure 4 – Segmentation Results', fontsize=16)
col_titles = ['Input Image', 'GT Mask (auto-generated)', 'Predicted Mask', 'Overlay (alpha=0.45)']
cmap_seg   = plt.cm.get_cmap('tab10', N_CLASSES)

for row in range(n_show):
    gray    = (X_test[row, :, :, 0] * 255).astype(np.uint8)
    gt_mask = y_test[row]
    pr_mask = y_pred_argmax[row]
    overlay = make_overlay(gray, pr_mask)

    data = [gray, gt_mask, pr_mask, overlay]
    cmaps = ['gray', cmap_seg, cmap_seg, None]

    for col, (d, cm_) in enumerate(zip(data, cmaps)):
        ax = axes[row, col]
        if cm_ is None:
            ax.imshow(d)
        else:
            ax.imshow(d, cmap=cm_, vmin=0, vmax=N_CLASSES-1)
        if row == 0:
            ax.set_title(col_titles[col], fontsize=11)
        ax.axis('off')

patches = [mpatches.Patch(color=cmap_seg(c), label=class_names[c]) for c in range(N_CLASSES)]
fig.legend(handles=patches, loc='lower center', ncol=4, fontsize=11)
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('/content/figure4_segmentation_results.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5 – Per-class Probability Maps (one test sample)
sample_idx = 0
sample_prob = model.predict(np.expand_dims(X_test[sample_idx], 0), verbose=0)[0]  # (H, W, 4)

fig, axes = plt.subplots(1, N_CLASSES + 1, figsize=(22, 5))
fig.suptitle('Figure 5 – Per-class Probability Maps (Test Sample)', fontsize=14)

gray = (X_test[sample_idx, :, :, 0] * 255).astype(np.uint8)
axes[0].imshow(gray, cmap='gray')
axes[0].set_title('Input Image')
axes[0].axis('off')

for c in range(N_CLASSES):
    im = axes[c + 1].imshow(sample_prob[:, :, c], cmap='hot', vmin=0, vmax=1)
    axes[c + 1].set_title(f'P(class {c}\n{class_names[c]})', fontsize=9)
    axes[c + 1].axis('off')
    fig.colorbar(im, ax=axes[c + 1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('/content/figure5_probability_maps.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Save final model weights
model.save('/content/drive/MyDrive/Retina_layer_semantic_segmentation/retina_3class_final.hdf5')
print('Model saved.')